# Summary

Crawl a bunch of sites and save data to GCP Cloud Storage

In [1]:
import sys, os, io

import pandas as pd
from tqdm import tqdm

# Web crawling and scraping tools class
sys.path.insert(0, "crawl_tools/")
import web_scraper as ws
import webfile_downloader as wfd
import web_crawler as wc

from google.cloud import storage


## Test the site scraper on one site

In [2]:
# turl = "https://en.wikipedia.org/wiki/Chaffey_College"
# turl = "https://www.ecs.org"
#
# # turl = "https://www.ecs.org/events/"
#
# test = await ws.webScraper.visit_page(url=turl)
#
# test.crawl_results.keys()
# type(test.crawl_results)

# print(test.crawl_results['atag_urls'])
# [u for u in test.crawl_results['atag_urls'] if u.find(".zip")>=0]
# test.crawl_results['ptag_text']

## Test file downloader on one site

In [3]:

turl = "https://nces.ed.gov/ipeds/datacenter/data/SFA2223.zip"
#
# # Note this downloads to a GCP bucket
# test = wfd.webFileDownloader(url=turl)
# test.download_document()
# t = test.read_document()

# t


## Web Crawler

## Crawl websites (and write results to GCS)

In [2]:
# Read crawl parameters
cp_filename = "crawl_configuraton.csv"
cp_path = "../data/crawler_params"
df_cp = pd.read_csv(filepath_or_buffer=os.path.join(cp_path, cp_filename))
df_cp = df_cp.fillna(" ")

df_cp



,seed_url,storage_folder,crawl_urls,dont_crawl_urls,crawl_depth,crawl_width
0,https://en.wikipedia.org/wiki/California_Commu...,wikipedia,,https://en.wikipedia.org/wiki/File:CDI-Seal-Co...,10,30
1,https://www.cccco.edu,cccco,,,10,30
2,https://www.ccleague.org,ccleague,,,10,30
3,https://cccaoe.org/,cccaoe,,,10,30
4,https://lao.ca.gov/Publications/,lao,https://lao.ca.gov/Publications/Report/4531;ht...,,1,5
5,https://ccrc.tc.columbia.edu/,columbia,,,10,30
6,https://nscresearchcenter.org/,nsc,,,10,30
7,https://www.aacc.nche.edu/,aacc,,,10,30
8,https://www.ecs.org/,ecs,,www.ecs.org/state/al/;www.ecs.org/state/ky/;ww...,10,30
9,https://nces.ed.gov/ipeds,ipeds,https://nces.ed.gov/ipeds/datacenter/data/SFA2...,,1,97


In [3]:
# Crawl seed_url and web links found on it and its child pages
dont_crawl_urls = []

for idx in tqdm(df_cp.index[-1:]):

    print("Crawling source: {}".format(df_cp.loc[idx, "storage_folder"]))

    crawler = wc.webCrawler(seed_url=df_cp.loc[idx, "seed_url"],
                            gcs_directory=df_cp.loc[idx, "storage_folder"])
    crlres = await crawler.crawl_sites(dont_crawl_urls=df_cp.loc[idx, "dont_crawl_urls"].split(";"),
                                       crawl_urls=df_cp.loc[idx, "crawl_urls"].split(";"),
                                       depth=df_cp.loc[idx, "crawl_depth"],
                                       width=df_cp.loc[idx, "crawl_width"])


  0%|          | 0/1 [00:00<?, ?it/s]

Crawling source: ipeds



100%|██████████| 1/1 [04:00<00:00, 240.65s/it]

Depth level finished: 1: 1 URLs crawled; 96 files downloaded; 60 URLs in to_crawl_urls; 195 URLs in dont_crawl_urls



100%|██████████| 1/1 [04:01<00:00, 241.16s/it]

Batch 1 saved to disk


## Read local crawl results

In [5]:


crawl_file = "aacc/webpages_pdfs/wwwaaccncheedu_2025Feb18_1.csv"
crawl_file1 = "ipeds/zipcsv_files/prep/descriptions_2025Feb20.csv"
crawl_file2 = "ipeds/webpages_pdfs/ncesedgov_2025Feb20_1.csv"


gcp_project_id = "eternal-bongo-435614-b9"
gcs_bucket_name = "ccc-crawl_data"

# Create a storage client
storage_client = storage.Client(project=gcp_project_id)

# Get the bucket
bucket = storage_client.bucket(gcs_bucket_name)

# Get the blob
blob = bucket.blob(crawl_file1)

# Works
data = blob.download_as_bytes()
dft1 = pd.read_csv(io.BytesIO(data))


# Get the blob
blob = bucket.blob(crawl_file2)

# Works
data = blob.download_as_bytes()
dft2 = pd.read_csv(io.BytesIO(data))




In [10]:
# Create a storage client
storage_client = storage.Client(project=gcp_project_id)

# Get the bucket
bucket = storage_client.bucket(gcs_bucket_name)

# Note: Client.list_blobs requires at least package version 1.17.0.
blobs = storage_client.list_blobs(bucket)

# Note: The call returns a response only when the iterator is consumed.
for blob in blobs:
    print(blob.name)

aacc/wwwaaccncheedu_2025Feb10_1.csv
aacc/wwwaaccncheedu_2025Feb10_2.csv
cccaoe/cccaoeorg_2025Feb10_1.csv
cccaoe/cccaoeorg_2025Feb10_2.csv
cccco/wwwccccoedu_2025Feb10_1.csv
cccco/wwwccccoedu_2025Feb10_3.csv
cccco/wwwccccoedu_2025Feb10_4.csv
ccleague/wwwccleagueorg_2025Feb10_1.csv
ccleague/wwwccleagueorg_2025Feb10_2.csv
columbia/ccrctccolumbiaedu_2025Feb10_1.csv
columbia/ccrctccolumbiaedu_2025Feb10_2.csv
ecs/wwwecsorg_2025Feb10_1.csv
ecs/wwwecsorg_2025Feb10_2.csv
ipeds/files/DRVEF2023.zip
ipeds/files/DRVOM2023.zip
ipeds/files/EAP2023_Dict.zip
ipeds/files/EF2023D.zip
ipeds/files/EFIA2023_Dict.zip
ipeds/files/GR200_23_Dict.zip
ipeds/files/GR2023_PELL_SSL_Dict.zip
ipeds/files/OM2023_Dict.zip
ipeds/files/S2023_SIS_Dict.zip
ipeds/files/SAL2023_NIS_Dict.zip
lao/laocagov_2025Feb10_1.csv
nsc/nscresearchcenterorg_2025Feb10_1.csv
wikipedia/enwikipediaorg_2025Feb10_1.csv
wikipedia/enwikipediaorg_2025Feb10_2.csv
wikipedia/enwikipediaorg_2025Feb10_3.csv
